In [1]:
import sys
import os
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from database import get_engine

print("All imports successful!")

All imports successful!


In [2]:
engine = get_engine()

df = pd.read_sql("""
    SELECT date, ticker, open, high, low, close, volume
    FROM stock_prices
    ORDER BY ticker, date
""", engine)

df['date'] = pd.to_datetime(df['date'])

print(f"Total rows loaded: {len(df):,}")
print(f"Total tickers: {df['ticker'].nunique():,}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
df.head(5)

Total rows loaded: 1,554,542
Total tickers: 579
Date range: 2014-01-02 to 2024-12-30


,date,ticker,open,high,low,close,volume
0,2014-01-02,A,36.878308,36.878308,36.264745,36.303497,2678848
1,2014-01-03,A,36.419745,37.039761,36.335783,36.762047,2609647
2,2014-01-06,A,37.072067,37.265823,36.529551,36.581219,2484665
3,2014-01-07,A,36.781431,37.220611,36.768514,37.104359,2045554
4,2014-01-08,A,37.026871,37.808357,36.923535,37.711479,3717981


In [3]:
def calculate_momentum_factors(df):
    grouped = df.groupby('ticker')['close']
    df['mom_12m']  = grouped.transform(lambda x: x.pct_change(252))
    df['mom_6m']   = grouped.transform(lambda x: x.pct_change(126))
    df['mom_1m']   = grouped.transform(lambda x: x.pct_change(21))
    df['mom_5d']   = grouped.transform(lambda x: x.pct_change(5))
    df['reversal'] = -df['mom_5d']
    return df

def calculate_volatility_factors(df):
    grouped_ret = df.groupby('ticker')['daily_return']
    df['vol_21d']   = grouped_ret.transform(lambda x: x.rolling(21).std() * np.sqrt(252))
    df['vol_63d']   = grouped_ret.transform(lambda x: x.rolling(63).std() * np.sqrt(252))
    df['vol_ratio'] = df['vol_21d'] / df['vol_63d']
    return df

def calculate_volume_factors(df):
    df['vol_avg_21d']  = df.groupby('ticker')['volume'].transform(lambda x: x.rolling(21).mean())
    df['volume_ratio'] = df['volume'] / df['vol_avg_21d']
    df['pvt'] = df.groupby('ticker').apply(
        lambda x: (x['daily_return'] * x['volume']).cumsum()
    ).reset_index(level=0, drop=True)
    return df

def calculate_mean_reversion_factors(df):
    df['high_52w']       = df.groupby('ticker')['close'].transform(lambda x: x.rolling(252).max())
    df['dist_from_high'] = (df['close'] - df['high_52w']) / df['high_52w']

    def compute_rsi(series, period=14):
        delta    = series.diff()
        gain     = delta.where(delta > 0, 0)
        loss     = -delta.where(delta < 0, 0)
        avg_gain = gain.rolling(period).mean()
        avg_loss = loss.rolling(period).mean()
        rs       = avg_gain / avg_loss
        return 100 - (100 / (1 + rs))

    df['rsi'] = df.groupby('ticker')['close'].transform(compute_rsi)
    return df

def calculate_all_factors(df):
    print("Sorting data...")
    df = df.sort_values(['ticker', 'date']).copy()

    print("Calculating daily returns...")
    df['daily_return'] = df.groupby('ticker')['close'].transform(lambda x: x.pct_change())

    print("Calculating momentum factors...")
    df = calculate_momentum_factors(df)

    print("Calculating volatility factors...")
    df = calculate_volatility_factors(df)

    print("Calculating volume factors...")
    df = calculate_volume_factors(df)

    print("Calculating mean reversion factors...")
    df = calculate_mean_reversion_factors(df)

    print("Done!")
    return df

print("All factor functions defined!")

All factor functions defined!


In [4]:
df = calculate_all_factors(df)

print(f"\nTotal rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")

Sorting data...
Calculating daily returns...
Calculating momentum factors...
Calculating volatility factors...
Calculating volume factors...
Calculating mean reversion factors...
Done!

Total rows: 1,554,542
Columns: ['date', 'ticker', 'open', 'high', 'low', 'close', 'volume', 'daily_return', 'mom_12m', 'mom_6m', 'mom_1m', 'mom_5d', 'reversal', 'vol_21d', 'vol_63d', 'vol_ratio', 'vol_avg_21d', 'volume_ratio', 'pvt', 'high_52w', 'dist_from_high', 'rsi']


In [5]:
FACTOR_COLS = [
    'date', 'ticker', 'close', 'daily_return',
    'mom_12m', 'mom_6m', 'mom_1m', 'mom_5d', 'reversal',
    'vol_21d', 'vol_63d', 'vol_ratio',
    'volume_ratio', 'pvt',
    'high_52w', 'dist_from_high', 'rsi'
]

factors_df = df[FACTOR_COLS].copy()
factors_df = factors_df.dropna()
factors_df['date'] = pd.to_datetime(factors_df['date']).dt.date

print(f"Rows to save: {len(factors_df):,}")
print(f"Date range: {factors_df['date'].min()} to {factors_df['date'].max()}")

factors_df.to_sql(
    'factors',
    engine,
    if_exists='replace',
    index=False,
    method='multi',
    chunksize=10000
)

print("All factors saved to database!")

Rows to save: 1,408,642
Date range: 2015-01-02 to 2024-12-30
All factors saved to database!
